# 23 — Build a Scalar Autograd Engine

**Master syllabus coverage:** V2 3.3–3.4

## Why this matters

Implementing reverse-mode autodiff removes the magic from `.backward()`. The important design ideas—dynamic graphs, topological order, local rules, and accumulation—carry directly to tensor frameworks.

## Learning objectives

- Represent scalar operations as a dynamic computation graph.
- Topologically order that graph for reverse traversal.
- Implement local backward rules with gradient accumulation.
- Verify a nonlinear neuron against PyTorch autograd.

Work through the notebook in order. Predict shapes and results before running each code cell. An assertion failure is part of the lesson: read it before changing code.


## 1. Build a scalar reverse-mode autograd engine

Each `Value` stores a scalar result, accumulated gradient, parent nodes, operation label,
and a local backward closure. Operators construct a directed acyclic graph during the
forward pass. Backward sorts parents before children, then visits that order in reverse.


In [ ]:
import math
import torch

class Value:
    def __init__(self, data, children=(), op="", label=""):
        self.data = float(data)
        self.grad = 0.0
        self._prev = set(children)
        self._op = op
        self.label = label
        self._backward = lambda: None

    def __repr__(self):
        return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")
        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out

    __radd__ = __add__

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), "*")
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    __rmul__ = __mul__

    def __pow__(self, exponent):
        assert isinstance(exponent, (int, float))
        out = Value(self.data**exponent, (self,), f"**{exponent}")
        def _backward():
            self.grad += exponent * self.data ** (exponent - 1) * out.grad
        out._backward = _backward
        return out

    def __neg__(self): return self * -1
    def __sub__(self, other): return self + (-other)
    def __rsub__(self, other): return other + (-self)
    def __truediv__(self, other): return self * other**-1

    def tanh(self):
        value = math.tanh(self.data)
        out = Value(value, (self,), "tanh")
        def _backward():
            self.grad += (1 - value**2) * out.grad
        out._backward = _backward
        return out

    def exp(self):
        value = math.exp(self.data)
        out = Value(value, (self,), "exp")
        def _backward():
            self.grad += value * out.grad
        out._backward = _backward
        return out

    def backward(self):
        order, visited = [], set()
        def visit(node):
            if node not in visited:
                visited.add(node)
                for parent in node._prev:
                    visit(parent)
                order.append(node)
        visit(self)
        self.grad = 1.0
        for node in reversed(order):
            node._backward()


## 2. Branching requires `+=`

The same node can appear through multiple paths. Every local backward function must
accumulate rather than assign. The topological traversal ensures a node receives all
downstream contributions before it propagates them further.


In [ ]:
x = Value(3.0, label="x")
y = x * x + 2 * x + 1
y.backward()
print("y:", y, "x:", x)
assert y.data == 16.0 and x.grad == 8.0  # derivative 2x + 2


## 3. A tiny neuron and squared loss

The engine is enough to construct a neuron, nonlinear activation, loss, and parameter
update. Real frameworks add tensors, broadcasting, efficient native kernels, graph
lifetime management, mixed precision, and many operations; the reverse-mode idea stays.


In [ ]:
inputs = [Value(2.0), Value(-1.0)]
weights = [Value(0.5, label="w0"), Value(-1.0, label="w1")]
bias = Value(0.25, label="b")
target = 0.5

preactivation = sum((weight * value for weight, value in zip(weights, inputs)), bias)
prediction = preactivation.tanh()
loss = (prediction - target) ** 2
loss.backward()
print("prediction:", prediction, "loss:", loss)
print("parameter gradients:", [weight.grad for weight in weights], bias.grad)


## 4. Verify every leaf against PyTorch

Independent implementations should agree in float64. This comparison catches local
derivative, graph ordering, and accumulation bugs more effectively than checking that
loss merely decreases once.


In [ ]:
tx = torch.tensor([2.0, -1.0], dtype=torch.float64)
tw = torch.tensor([0.5, -1.0], dtype=torch.float64, requires_grad=True)
tb = torch.tensor(0.25, dtype=torch.float64, requires_grad=True)
tp = torch.tanh((tx * tw).sum() + tb)
tloss = (tp - target) ** 2
tloss.backward()

assert math.isclose(loss.data, tloss.item(), rel_tol=1e-12)
for manual, reference in zip(weights, tw.grad):
    assert math.isclose(manual.grad, reference.item(), rel_tol=1e-12)
assert math.isclose(bias.grad, tb.grad.item(), rel_tol=1e-12)
print("forward values and all parameter gradients match PyTorch")


## Failure modes to recognize

- **Gradient assignment instead of accumulation:** branch contributions disappear.
- **Forward-order backward pass:** parents propagate before receiving all contributions.
- **Unseeded output:** every gradient remains zero.
- **Stale graph gradients:** repeated backward calls accumulate unless deliberately cleared.

These are diagnostic patterns, not trivia. When a future model fails, reduce the problem to the smallest example in this notebook and test the relevant invariant.


## Deliberate practice

1. Add `log`, `relu`, and sigmoid operations with local derivative tests.
2. Implement a method that clears gradients for all nodes reachable from an output.
3. Train the tiny neuron for 20 steps and compare the loss trajectory with PyTorch.

Do not merely rerun the provided cells. Make a prediction, change one variable, and write down why the result did or did not match your prediction.

## Exit ticket

**You are ready to continue when:** you can implement a new differentiable scalar operation, state its local rule, and verify branched gradients against PyTorch.

**Next:** 24 — Tensor autograd and gradient checking.
